In [1]:
import os
import sys
import math
import time
import logging

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay
from tensorflow.keras.layers import Dense, Dropout, Flatten, LeakyReLU
from tensorflow.keras.models import load_model

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool

from mlpng import Core
from mlpng.utils import rmse_metrics, get_data
from mlpng.utils.dataloaders import KappaDataset

# Setup GPU strategy
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
strategy = tf.distribute.MirroredStrategy()

plt.style.use("seaborn-v0_8-paper")
print(
    f"TF {tf.__version__}, GPUs: {len(gpus)}, Strategy: {strategy.num_replicas_in_sync} replicas"
)

TF 2.15.1, GPUs: 1, Strategy: 1 replicas


In [2]:
# Configuration
core = Core(["./settings/n256.json", "--shapes", "local", "--nsims", "10000"])
print(f"nside={core.nside}, npix={core.npix}, shapes={core.shapes}")

# Training settings
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
DATA_FRACTION = 0.1
BASE_SPLIT = np.array([0.8, 0.1, 0.1], dtype=np.float32)
DUPLICATES = [25, 10, 10]
PHI_SCALES = [0, 1, 10, 100, 1000]

# Simplified best parameters from Optuna (transformer_levels=0, so no transformer)
best_params = {
    "initial_lr": 0.006063500256249086,
    "pool_p": 2,
    "dropout_rate": 0.15,
    "weight_decay": 3.78e-07,
    "dense_units": 32,
    "dense_layers": 1,
}

# Paths
cache_dir = os.environ.get("SCRATCH", "/tmp") + "/tf_cache"
model_dir = f"{core.dirs['model']}/{core.name}"
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Compute steps
fractions = BASE_SPLIT * DATA_FRACTION
approx_samples = max(1, int(core.total_sims * fractions[0])) * DUPLICATES[0]
steps_per_epoch = max(1, math.ceil(approx_samples / BATCH_SIZE))
decay_steps = steps_per_epoch * 2
print(f"Steps/epoch: {steps_per_epoch}, Decay steps: {decay_steps}")

03-Dec-25 09:43:01 - mlpng.core - DEBUG - Parsing CLI args: ['./settings/n256.json', '--shapes', 'local', '--nsims', '10000']
03-Dec-25 09:43:01 - mlpng.core - INFO - Loading settings from file './settings/n256.json'
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Forcing setting 'nsims' to 10000 due to CLI
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Forcing setting 'shapes' to ['local'] due to CLI
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Forcing setting 'tf_cache' to True due to CLI
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Forcing setting 'tf_mem_cache' to True due to CLI
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.105e-09
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.965
03-Dec-25 09:43:01 - mlpng.core - DEBUG - Running with settings: 
{
  "cosmo_params": {
    "As": 2.105e-09,
    "ns": 0.965,
    "pivot_scalar": 0.05,
    "H0": 67.4,
    "ombh2": 0.0224,
    "omch2": 0.12,
    "tau": 0.054
  },
  "nsims

## Simplified Encoder Model (No Transformer)

In [ ]:
class EncoderBlock(tf.keras.layers.Layer):
    """Simplified encoder block: Conv + Dropout + Pool."""

    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        K,
        pool_p,
        max_batch_size,
        dropout_rate=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.config_dict = dict(
            nside=nside,
            npix=npix,
            fin=fin,
            fout=fout,
            K=K,
            pool_p=pool_p,
            max_batch_size=max_batch_size,
            dropout_rate=dropout_rate,
        )
        layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation="gelu",
                use_bn=True,
                use_bias=True,
                initializer=tf.keras.initializers.HeNormal(),
            ),
            Dropout(dropout_rate),
            HealpyPool(pool_p, "AVG"),
        ]
        self.body = HealpyGCNN(
            nside=nside,
            indices=np.arange(npix),
            layers=layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

    def call(self, x, training=False):
        return self.body(x, training=training)

    def get_config(self):
        return {**super().get_config(), **self.config_dict}


def build_encoder_model(npix, npols, n_outputs, params):
    """Build simplified encoder fnl model."""
    nside = hp.npix2nside(npix)
    pool_p = params["pool_p"]
    nside_factor = 2**pool_p
    depth = int(math.log(nside, nside_factor))

    level_nsides = [nside // (nside_factor**i) for i in range(depth + 1)]
    level_npixels = [12 * ns**2 for ns in level_nsides]
    channels = [npols] + [2 ** (i + 5) for i in range(depth + 1)]

    inputs = tf.keras.Input(shape=(npix, npols), name="input")
    x = inputs

    for i in range(depth):
        x = EncoderBlock(
            level_nsides[i],
            level_npixels[i],
            fin=channels[i],
            fout=channels[i + 1],
            K=3,
            pool_p=pool_p,
            max_batch_size=BATCH_SIZE,
            dropout_rate=params["dropout_rate"],
        )(x)

    x = Flatten()(x)
    for _ in range(params["dense_layers"]):
        x = Dense(
            params["dense_units"], activation="gelu", kernel_initializer="he_normal"
        )(x)
    outputs = Dense(n_outputs, kernel_initializer="he_normal")(x)

    return tf.keras.Model(inputs, outputs, name="encoder_fnl")


# Register custom object for model loading
tf.keras.utils.get_custom_objects()["EncoderBlock"] = EncoderBlock

## Jorik SCN Model

In [4]:
def build_jorik_model(npix, npols, n_outputs):
    """Jorik SCN model for fnl estimation."""
    nside = hp.npix2nside(npix)
    n_layers = math.floor(math.log(nside, 2))

    layers = []
    for _ in range(n_layers):
        layers.append(
            HealpyChebyshev(K=2, Fout=32, use_bn=True, activation=LeakyReLU(0.3))
        )
        layers.append(Dropout(0.1))
        layers.append(HealpyPool(1, "AVG"))

    layers.append(Flatten())
    layers.append(Dropout(0.3))
    layers.append(Dense(32, activation=LeakyReLU(0.3)))
    layers.append(Dense(32, activation=LeakyReLU(0.3)))
    layers.append(Dense(n_outputs))

    model = HealpyGCNN(
        nside,
        indices=np.arange(npix),
        layers=layers,
        n_neighbors=8,
        max_batch_size=BATCH_SIZE,
        initial_Fin=npols,
    )
    model.build((None, npix, npols))
    return model

## Helper Functions

In [5]:
def create_dataset(x_output, phi_scale=None, cache_suffix=""):
    """Create train/val/test datasets."""
    kwargs = {"x_output": x_output, "y_output": "fnl"}
    if phi_scale is not None:
        kwargs["phi_scale"] = phi_scale
    ds = KappaDataset.fromCore(core, **kwargs)
    return ds.split(
        train_size=float(fractions[0]),
        val_size=float(fractions[1]),
        test_size=float(fractions[2]),
        to_tf=True,
        batch_size=BATCH_SIZE,
        duplicates=DUPLICATES,
        cache_dir=cache_dir,
        cache_file=f"phi-impact-n{core.nside}{cache_suffix}",
        gen_batch_size=16,
    )


def cache_dataset(train_ds, val_ds, test_ds, name=""):
    """Iterate through datasets to populate cache."""
    for ds, split in [(train_ds, "train"), (val_ds, "val"), (test_ds, "test")]:
        for _ in tqdm(ds, desc=f"Caching {name} {split}", leave=False):
            pass


def train_model(model, train_ds, val_ds, use_cosine=True, verbose=1):
    """Train model and return history with timing."""
    lr = best_params["initial_lr"] if use_cosine else 1e-3
    if use_cosine:
        lr_schedule = CosineDecayRestarts(
            lr, decay_steps, t_mul=2.0, m_mul=0.95, alpha=0.01
        )
    else:
        lr_schedule = ExponentialDecay(lr, decay_steps, decay_rate=0.96, staircase=True)

    with strategy.scope():
        optimizer = AdamW(
            learning_rate=lr_schedule,
            weight_decay=best_params.get("weight_decay", 1e-6),
        )
        model.compile(
            optimizer=optimizer, loss="mse", metrics=rmse_metrics(core.shapes)
        )

    callbacks = [
        TerminateOnNaN(),
        EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    ]

    start = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        steps_per_epoch=steps_per_epoch,
        callbacks=callbacks,
        verbose=verbose,
    )
    train_time = time.time() - start
    return history, train_time


def evaluate_model(model, test_ds):
    """Evaluate model and return predictions, truth, and metrics."""
    preds = model.predict(test_ds, verbose=0)
    truth = np.concatenate([y.numpy() for _, y in test_ds])
    if len(core.shapes) == 1:
        truth, preds = truth.ravel(), preds.ravel()
    rmse = np.sqrt(np.mean((preds - truth) ** 2))
    mae = np.mean(np.abs(preds - truth))
    return preds, truth, rmse, mae


def load_or_train(model_path, build_fn, train_ds, val_ds, use_cosine=True):
    """Load model if exists, otherwise train and save."""
    if os.path.exists(model_path):
        print(f"Loading {model_path}")
        with strategy.scope():
            return load_model(model_path), None, 0

    with strategy.scope():
        model = build_fn()
    history, train_time = train_model(model, train_ds, val_ds, use_cosine=use_cosine)
    model.save(model_path)
    print(f"Saved {model_path}")
    return model, history, train_time


def get_fisher_sigma(lensed=True):
    """Get Fisher sigma from data file."""
    try:
        l_str = "lensed" if lensed else "unlensed"
        fisher = get_data(core.file, f"fisher/{l_str}/{core.shapes[0]}", 0)
        return 1 / np.sqrt(fisher)
    except:
        return None

## Train Jorik Models (Unlensed & Lensed)

In [6]:
results = {}  # Store all results

# Jorik Unlensed
train_u, val_u, test_u = create_dataset("unlensed", cache_suffix="-unlensed")
cache_dataset(train_u, val_u, test_u, "unlensed")

jorik_unlensed, hist_ju, time_ju = load_or_train(
    f"{model_dir}/jorik-unlensed.keras",
    lambda: build_jorik_model(core.npix, core.npols, len(core.shapes)),
    train_u,
    val_u,
    use_cosine=False,
)
preds_ju, truth_ju, rmse_ju, mae_ju = evaluate_model(jorik_unlensed, test_u)
results["jorik_unlensed"] = {
    "model": jorik_unlensed,
    "history": hist_ju,
    "time": time_ju,
    "preds": preds_ju,
    "truth": truth_ju,
    "rmse": rmse_ju,
    "mae": mae_ju,
}
print(f"Jorik Unlensed - RMSE: {rmse_ju:.4f}, MAE: {mae_ju:.4f}, Time: {time_ju:.1f}s")

03-Dec-25 09:43:57 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_10000_p1.0.hdf5'...
03-Dec-25 09:43:57 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00872105]
03-Dec-25 09:43:57 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.708177]
03-Dec-25 09:43:58 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432481, std div: 10.88985972354282
03-Dec-25 09:43:58 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
03-Dec-25 09:43:58 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-unlensed-train.cache'
03-Dec-25 09:44:07 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-unlensed-val.cache'
03-Dec-25 09:44:11 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/clie

Caching unlensed train:   0%|          | 0/625 [00:00<?, ?it/s]

Caching unlensed val:   0%|          | 0/32 [00:00<?, ?it/s]

Caching unlensed test:   0%|          | 0/32 [00:00<?, ?it/s]

Loading data/models/l767_n256_T_10000_p1.0/jorik-unlensed.keras
Jorik Unlensed - RMSE: 41.1140, MAE: 33.6951, Time: 0.0s


In [ ]:
# Jorik Lensed (phi_scale=1)
train_l, val_l, test_l = create_dataset(
    "lensed", phi_scale=1.0, cache_suffix="-lensed-phi1"
)
cache_dataset(train_l, val_l, test_l, "lensed-phi1")

jorik_lensed, hist_jl, time_jl = load_or_train(
    f"{model_dir}/jorik-lensed.keras",
    lambda: build_jorik_model(core.npix, core.npols, len(core.shapes)),
    train_l,
    val_l,
    use_cosine=False,
)
preds_jl, truth_jl, rmse_jl, mae_jl = evaluate_model(jorik_lensed, test_l)
results["jorik_lensed"] = {
    "model": jorik_lensed,
    "history": hist_jl,
    "time": time_jl,
    "preds": preds_jl,
    "truth": truth_jl,
    "rmse": rmse_jl,
    "mae": mae_jl,
}
print(f"Jorik Lensed - RMSE: {rmse_jl:.4f}, MAE: {mae_jl:.4f}, Time: {time_jl:.1f}s")

03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_10000_p1.0.hdf5'...
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00872105]
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.708177]
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432481, std div: 10.88985972354282
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00872105]
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.708177]
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432481, std div: 10.88985972354282
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
03-Dec-25 09:51:09 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/ph

## Train CNN Range Model (phi_scale 0-1000)

In [9]:
train_r, val_r, test_r = create_dataset(
    "lensed", phi_scale=(0, 1000), cache_suffix="-range"
)
cache_dataset(train_r, val_r, test_r, "range")

cnn_range, hist_cr, time_cr = load_or_train(
    f"{model_dir}/encoder-range.keras",
    lambda: build_encoder_model(core.npix, core.npols, len(core.shapes), best_params),
    train_r,
    val_r,
    use_cosine=True,
)
preds_cr, truth_cr, rmse_cr, mae_cr = evaluate_model(cnn_range, test_r)
results["cnn_range"] = {
    "model": cnn_range,
    "history": hist_cr,
    "time": time_cr,
    "preds": preds_cr,
    "truth": truth_cr,
    "rmse": rmse_cr,
    "mae": mae_cr,
}
print(f"CNN Range - RMSE: {rmse_cr:.4f}, MAE: {mae_cr:.4f}, Time: {time_cr:.1f}s")

02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_10000_p1.0.hdf5'...
02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00872105]
02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.708177]
02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432481, std div: 10.88985972354282
02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
02-Dec-25 18:24:28 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-range-train.cache'
02-Dec-25 18:27:49 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-range-val.cache'
02-Dec-25 18:29:10 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/use

Caching range train:   0%|          | 0/625 [00:00<?, ?it/s]

Caching range val:   0%|          | 0/32 [00:00<?, ?it/s]

Caching range test:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1/100
625/625 [==============================] - 192s 302ms/step - loss: 45834.3945 - rmse_local: 214.0897 - val_loss: 13903.4678 - val_rmse_local: 117.9130
Epoch 2/100
625/625 [==============================] - 189s 302ms/step - loss: 16810.0508 - rmse_local: 129.6536 - val_loss: 8186.7900 - val_rmse_local: 90.4809
Epoch 3/100
625/625 [==============================] - 188s 301ms/step - loss: 23825.6953 - rmse_local: 154.3557 - val_loss: 16005.8809 - val_rmse_local: 126.5144
Epoch 4/100
625/625 [==============================] - 188s 301ms/step - loss: 16782.9414 - rmse_local: 129.5490 - val_loss: 10425.5576 - val_rmse_local: 102.1056
Epoch 5/100
625/625 [==============================] - 188s 301ms/step - loss: 14332.9727 - rmse_local: 119.7204 - val_loss: 7046.4111 - val_rmse_local: 83.9429
Epoch 6/100
625/625 [==============================] - 188s 301ms/step - loss: 11993.9365 - rmse_local: 109.5168 - val_loss: 6173.3145 - val_rmse_local: 78.5704
Epoch 7/100
625/625 [=======

## Train Phi-Specific Models

In [ ]:
phi_models = {}

for phi in tqdm(PHI_SCALES, desc="Training phi-specific models"):
    train_p, val_p, test_p = create_dataset(
        "lensed", phi_scale=float(phi), cache_suffix=f"-phi{phi}"
    )
    cache_dataset(train_p, val_p, test_p, f"phi{phi}")

    model_p, hist_p, time_p = load_or_train(
        f"{model_dir}/encoder-phi{phi}.keras",
        lambda: build_encoder_model(
            core.npix, core.npols, len(core.shapes), best_params
        ),
        train_p,
        val_p,
        use_cosine=True,
    )

    preds_p, truth_p, rmse_p, mae_p = evaluate_model(model_p, test_p)
    phi_models[phi] = {
        "model": model_p,
        "history": hist_p,
        "time": time_p,
        "preds": preds_p,
        "truth": truth_p,
        "rmse": rmse_p,
        "mae": mae_p,
        "test_ds": test_p,
    }
    print(f"phi={phi}: RMSE={rmse_p:.4f}, MAE={mae_p:.4f}, Time={time_p:.1f}s")

Training phi-specific models:   0%|          | 0/5 [00:00<?, ?it/s]

02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_10000_p1.0.hdf5'...
02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00872105]
02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.708177]
02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432481, std div: 10.88985972354282
02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
02-Dec-25 20:12:59 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-phi0-train.cache'
02-Dec-25 20:16:04 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/phi-impact-n256-phi0-val.cache'
02-Dec-25 20:17:17 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users

Caching phi0 train:   0%|          | 0/625 [00:00<?, ?it/s]

Caching phi0 val:   0%|          | 0/32 [00:00<?, ?it/s]

Caching phi0 test:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1/100
625/625 [==============================] - 192s 302ms/step - loss: 17705.0352 - rmse_local: 133.0603 - val_loss: 4613.9316 - val_rmse_local: 67.9259
Epoch 2/100
625/625 [==============================] - 188s 301ms/step - loss: 9359.2539 - rmse_local: 96.7432 - val_loss: 1255.0468 - val_rmse_local: 35.4266
Epoch 3/100
625/625 [==============================] - 188s 301ms/step - loss: 11298.4531 - rmse_local: 106.2942 - val_loss: 24919.0293 - val_rmse_local: 157.8576
Epoch 4/100
625/625 [==============================] - 189s 302ms/step - loss: 9608.2266 - rmse_local: 98.0216 - val_loss: 16853.7188 - val_rmse_local: 129.8219
Epoch 5/100
625/625 [==============================] - 188s 301ms/step - loss: 9471.7441 - rmse_local: 97.3229 - val_loss: 3996.2415 - val_rmse_local: 63.2158
Epoch 6/100
625/625 [==============================] - 189s 301ms/step - loss: 7767.0728 - rmse_local: 88.1310 - val_loss: 1003.1373 - val_rmse_local: 31.6723
Epoch 7/100
625/625 [=================

Caching phi1 train:   0%|          | 0/625 [00:00<?, ?it/s]

Caching phi1 val:   0%|          | 0/32 [00:00<?, ?it/s]

Caching phi1 test:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1/100
625/625 [==============================] - 192s 302ms/step - loss: 19246.5469 - rmse_local: 138.7319 - val_loss: 20343.0742 - val_rmse_local: 142.6292
Epoch 2/100
625/625 [==============================] - 188s 301ms/step - loss: 9493.4961 - rmse_local: 97.4346 - val_loss: 1531.1360 - val_rmse_local: 39.1297
Epoch 3/100
625/625 [==============================] - 188s 301ms/step - loss: 10872.3984 - rmse_local: 104.2708 - val_loss: 4168.5508 - val_rmse_local: 64.5643
Epoch 4/100
625/625 [==============================] - 188s 301ms/step - loss: 9561.2959 - rmse_local: 97.7819 - val_loss: 19291.1738 - val_rmse_local: 138.8927
Epoch 5/100
625/625 [==============================] - 188s 301ms/step - loss: 7908.3682 - rmse_local: 88.9290 - val_loss: 3909.0835 - val_rmse_local: 62.5227
Epoch 6/100
625/625 [==============================] - 188s 301ms/step - loss: 6695.8809 - rmse_local: 81.8284 - val_loss: 911.1089 - val_rmse_local: 30.1846
Epoch 7/100
625/625 [==================

## Impact Plot: RMSE vs Phi-Scale

In [ ]:
# Evaluate all models at each phi_scale
impact = {"jorik_unlensed": [], "jorik_lensed": [], "cnn_range": [], "phi_specific": []}

for phi in PHI_SCALES:
    test_ds = phi_models[phi]["test_ds"]

    # Jorik unlensed (use unlensed test set for phi=0, otherwise lensed)
    if phi == 0:
        _, _, rmse, _ = evaluate_model(jorik_unlensed, test_u)
    else:
        _, _, rmse, _ = evaluate_model(jorik_unlensed, test_ds)
    impact["jorik_unlensed"].append(rmse)

    # Jorik lensed
    _, _, rmse, _ = evaluate_model(jorik_lensed, test_ds)
    impact["jorik_lensed"].append(rmse)

    # CNN range
    _, _, rmse, _ = evaluate_model(cnn_range, test_ds)
    impact["cnn_range"].append(rmse)

    # Phi-specific
    impact["phi_specific"].append(phi_models[phi]["rmse"])

# Get Fisher bounds
sigma_unlensed = get_fisher_sigma(lensed=False)
sigma_lensed = get_fisher_sigma(lensed=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(PHI_SCALES))

ax.plot(x, impact["jorik_unlensed"], "s-", label="Jorik (unlensed)", markersize=8)
ax.plot(x, impact["jorik_lensed"], "o-", label="Jorik (lensed)", markersize=8)
ax.plot(x, impact["cnn_range"], "^-", label="CNN Range (0-1000)", markersize=8)
ax.plot(x, impact["phi_specific"], "D-", label="Phi-Specific", markersize=8)

if sigma_unlensed:
    ax.axhline(
        sigma_unlensed,
        color="green",
        linestyle=":",
        lw=2,
        label=f"Fisher σ unlensed ({sigma_unlensed:.2f})",
    )
if sigma_lensed:
    ax.axhline(
        sigma_lensed,
        color="red",
        linestyle=":",
        lw=2,
        label=f"Fisher σ lensed ({sigma_lensed:.2f})",
    )

ax.set_xticks(x)
ax.set_xticklabels(PHI_SCALES)
ax.set_xlabel("phi_scale")
ax.set_ylabel("RMSE")
ax.set_title(f"fnl RMSE vs phi_scale (nside={core.nside})")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Scatter Plots: Predictions vs Truth

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sigma = sigma_lensed or 50

plot_data = [
    ("Jorik Unlensed", results["jorik_unlensed"]),
    ("Jorik Lensed", results["jorik_lensed"]),
    ("CNN Range", results["cnn_range"]),
    ("Phi=0", phi_models[0]),
    ("Phi=100", phi_models[100]),
    ("Phi=1000", phi_models[1000]),
]

for ax, (name, data) in zip(axes.flat, plot_data):
    truth, preds = data["truth"], data["preds"]
    line = np.array([np.nanmin(truth), np.nanmax(truth)])
    ax.scatter(truth, preds, alpha=0.5, s=10)
    ax.plot(line, line, "r--", lw=2, label="Perfect")
    ax.fill_between(
        line,
        line - sigma,
        line + sigma,
        alpha=0.2,
        color="green",
        label=f"±σ={sigma:.1f}",
    )
    ax.set_xlabel("True fnl")
    ax.set_ylabel("Predicted fnl")
    ax.set_title(f"{name}\nRMSE={data['rmse']:.2f}")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (name, data) in zip(axes.flat, plot_data):
    if data["history"] is not None:
        ax.plot(data["history"].history["loss"], label="Train")
        ax.plot(data["history"].history["val_loss"], label="Val")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss (MSE)")
        ax.legend()
    else:
        ax.text(
            0.5,
            0.5,
            "Loaded from file",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Inference Error vs Phi-Scale

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(PHI_SCALES))
width = 0.2

bars1 = ax.bar(x - 1.5 * width, impact["jorik_unlensed"], width, label="Jorik Unlensed")
bars2 = ax.bar(x - 0.5 * width, impact["jorik_lensed"], width, label="Jorik Lensed")
bars3 = ax.bar(x + 0.5 * width, impact["cnn_range"], width, label="CNN Range")
bars4 = ax.bar(x + 1.5 * width, impact["phi_specific"], width, label="Phi-Specific")

if sigma_lensed:
    ax.axhline(
        sigma_lensed,
        color="red",
        linestyle=":",
        lw=2,
        label=f"Fisher σ ({sigma_lensed:.2f})",
    )

ax.set_xticks(x)
ax.set_xticklabels(PHI_SCALES)
ax.set_xlabel("phi_scale")
ax.set_ylabel("RMSE")
ax.set_title(f"Inference Error vs phi_scale (nside={core.nside})")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Add value labels
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:.1f}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=7,
        )

plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
print("\nSummary: Model Performance")
print("=" * 70)
print(f"{'Model':<20} {'RMSE':<10} {'MAE':<10} {'Time (s)':<10} {'Val Loss':<12}")
print("-" * 70)

for name, data in [
    ("Jorik Unlensed", results["jorik_unlensed"]),
    ("Jorik Lensed", results["jorik_lensed"]),
    ("CNN Range", results["cnn_range"]),
]:
    val_loss = min(data["history"].history["val_loss"]) if data["history"] else "N/A"
    print(
        f"{name:<20} {data['rmse']:<10.4f} {data['mae']:<10.4f} {data['time']:<10.1f} {val_loss if isinstance(val_loss, str) else f'{val_loss:.6f}':<12}"
    )

for phi in PHI_SCALES:
    data = phi_models[phi]
    val_loss = min(data["history"].history["val_loss"]) if data["history"] else "N/A"
    print(
        f"Phi={phi:<14} {data['rmse']:<10.4f} {data['mae']:<10.4f} {data['time']:<10.1f} {val_loss if isinstance(val_loss, str) else f'{val_loss:.6f}':<12}"
    )

print("=" * 70)
if sigma_unlensed:
    print(f"Fisher σ (unlensed): {sigma_unlensed:.4f}")
if sigma_lensed:
    print(f"Fisher σ (lensed): {sigma_lensed:.4f}")